# Module 03 — AI Agents
## Lesson 8 — Human-in-the-Loop

**Core principle:** the model can recommend or propose; human approval grants authority for a specific action.

The examples are deterministic and simulated. No real purchase, email, or production change is performed.


## Human-in-the-loop patterns

- **Approval before action:** gate purchases, sends, deployments, deletes, or other consequential side effects.
- **Clarification:** ask when required intent or information is ambiguous.
- **Escalation:** hand off when risk, confidence, policy, or permissions require human judgment.
- **Review after drafting:** allow preparation but require human review before publication/execution.

Use humans at meaningful authority or judgment boundaries, not mechanically after every agent step.


## Approval should bind to an exact action

The host validates the proposal, applies deterministic risk policy, creates an approval request for the exact action, persists a checkpoint, and resumes only after a decision arrives.


In [ ]:
from dataclasses import dataclass
from enum import Enum
import hashlib
import json
from typing import Any

class RiskLevel(str, Enum):
    LOW = "low"
    HIGH = "high"

@dataclass(frozen=True)
class ActionProposal:
    tool: str
    arguments: dict[str, Any]
    summary: str
    risk: RiskLevel

def proposal_fingerprint(proposal: ActionProposal) -> str:
    payload = {"tool": proposal.tool, "arguments": proposal.arguments}
    canonical = json.dumps(payload, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()

proposal = ActionProposal(
    tool="book_hotel",
    arguments={
        "city": "Melbourne",
        "hotel": "Harbour View",
        "nights": 1,
        "total_aud": 310,
        "refundable": False,
    },
    summary="Book one non-refundable night for A$310.",
    risk=RiskLevel.HIGH,
)

print(proposal_fingerprint(proposal)[:12])


## Deterministic risk policy

The model should not decide whether its own proposal requires approval. The host uses policy based on tool type, risk, cost, environment, or role.


In [ ]:
READ_ONLY_TOOLS = {"search_hotels", "get_weather", "get_sun_times"}
SIDE_EFFECT_TOOLS = {"book_hotel", "send_email", "deploy_release", "delete_record"}

def requires_approval(proposal: ActionProposal) -> bool:
    if proposal.tool in SIDE_EFFECT_TOOLS:
        return True
    if proposal.risk == RiskLevel.HIGH:
        return True
    return False

print(requires_approval(proposal))


## Create a resumable approval request

Approval state belongs to the application. Record an ID, exact proposal fingerprint, status, decision identity, and optional note.


In [ ]:
class ApprovalStatus(str, Enum):
    PENDING = "pending"
    APPROVED = "approved"
    DENIED = "denied"

@dataclass
class ApprovalRequest:
    id: str
    proposal: ActionProposal
    proposal_hash: str
    status: ApprovalStatus = ApprovalStatus.PENDING
    decided_by: str | None = None
    note: str | None = None

def create_approval_request(request_id: str, proposal: ActionProposal) -> ApprovalRequest:
    if not requires_approval(proposal):
        raise ValueError("This proposal does not require approval")
    return ApprovalRequest(
        id=request_id,
        proposal=proposal,
        proposal_hash=proposal_fingerprint(proposal),
    )

approval = create_approval_request("approval-001", proposal)
approval


## Pause/resume instead of waiting in memory

A human may respond much later. Persist the agent checkpoint and return `WAITING_FOR_APPROVAL`; do not leave an in-memory loop sleeping for hours.

A checkpoint should include current goal/state, exact proposal, approval request, budgets/expiry, and enough information to safely resume.


## Approval, denial, and modification

A person may deny the action or request different constraints. Treat modification as a new proposal rather than mutating an already reviewed action.


In [ ]:
def decide(
    request: ApprovalRequest,
    *,
    status: ApprovalStatus,
    decided_by: str,
    note: str | None = None,
) -> None:
    if request.status != ApprovalStatus.PENDING:
        raise ValueError("Approval request has already been decided")
    request.status = status
    request.decided_by = decided_by
    request.note = note

decide(
    approval,
    status=ApprovalStatus.DENIED,
    decided_by="demo-user",
    note="Find a refundable option under A$250.",
)
print(approval.status, approval.note)


## Execute only the exact approved proposal

The fingerprint prevents an approval for one action from being reused after arguments silently change. A production system should additionally revalidate price, permissions, resource version, environment, and expiry before execution.


In [ ]:
def simulated_book_hotel(arguments: dict[str, Any]) -> dict[str, Any]:
    return {"booking_id": "SIM-1234", "status": "booked", **arguments}

def execute_approved(
    request: ApprovalRequest,
    proposal_to_execute: ActionProposal,
) -> dict[str, Any]:
    if request.status != ApprovalStatus.APPROVED:
        raise PermissionError("Action has not been approved")
    if proposal_fingerprint(proposal_to_execute) != request.proposal_hash:
        raise PermissionError("Proposal changed after approval")
    if proposal_to_execute.tool != "book_hotel":
        raise ValueError("Unsupported demo action")
    return simulated_book_hotel(proposal_to_execute.arguments)


## Human review does not replace deterministic safety

Reject invalid, forbidden, or unauthorised proposals before creating an approval request. Human approval should sit after authentication, schema validation, tenant isolation, and hard policy checks—not compensate for their absence.


## What should an approval UI show?

Show the exact action, important arguments, side effects, cost/scope, relevant evidence, why approval is required, and clear approve/deny controls. The reviewer should not need hidden prompts or raw chain-of-thought.


## Exercises

1. Approve the hotel proposal and execute the simulated booking.
2. Change `total_aud` after approval and verify the fingerprint blocks execution.
3. Add `expires_at` and reject stale approvals.
4. Adapt the pattern to `send_email`: drafting is automatic, sending requires approval.
5. Design the visible approval fields for a production rollback.
6. Model a `WAITING_FOR_APPROVAL` checkpoint rather than keeping the agent loop alive.

Next: **Lesson 9 — Deterministic Workflows vs Agents**.
